## 2点相関関数（Landy-Szalay推定量）

Landy-Szalay (1993) 推定量は以下で定義される：

$$\hat{\xi}(r) = \frac{DD(r) - 2\,DR(r) + RR(r)}{RR(r)}$$

各項は正規化されたペアカウント：

$$DD(r) = \frac{n_{DD}(r)}{N_D(N_D-1)/2}, \quad DR(r) = \frac{n_{DR}(r)}{N_D N_R}, \quad RR(r) = \frac{n_{RR}(r)}{N_R(N_R-1)/2}$$

- $N_D$：データ点数（Colesの店舗数）
- $N_R$：ランダムカタログのサイズ（$N_R \gg N_D$）
- $r$：ハーバーサイン距離 [km]

ランダムカタログはオーストラリアのバウンディングボックス内で一様サンプリング。
Landy-Szalay推定量の分散の下限（Poisson）は $\text{Var}[\hat{\xi}] \approx (1 + \hat{\xi})^2 / n_{RR}$。

In [20]:
%use dataframe
%use lets-plot

In [21]:
import kotlin.math.*
import kotlin.random.Random

// --- データ読み込み ---
val df = DataFrame.readCSV("./output/coles_locations.csv")

data class Point(val lat: Double, val lon: Double)

val dataPoints: List<Point> = df.rows().map { row ->
    Point(lat = row["lat"] as Double, lon = row["lon"] as Double)
}
val nD = dataPoints.size
println("N_D = $nD")

N_D = 685


In [22]:
// --- ハーバーサイン距離 [km] ---
fun haversine(p1: Point, p2: Point): Double {
    val R = 6371.0
    val dLat = Math.toRadians(p2.lat - p1.lat)
    val dLon = Math.toRadians(p2.lon - p1.lon)
    val sinLat = sin(dLat / 2)
    val sinLon = sin(dLon / 2)
    val a = sinLat * sinLat +
            cos(Math.toRadians(p1.lat)) * cos(Math.toRadians(p2.lat)) * sinLon * sinLon
    return 2.0 * R * asin(sqrt(a))
}

In [23]:
// --- 平均最近傍距離の計算（ビン設計の基準スケール）---
// O(N^2) だが N=685 なので数秒で完了
val nnDistances = dataPoints.map { p1 ->
    dataPoints.filter { it !== p1 }.minOf { p2 -> haversine(p1, p2) }
}
val meanNN = nnDistances.average()
val minNN  = nnDistances.min()
val maxNN  = nnDistances.max()

println("最近傍距離 [km]:")
println("  平均 (mean) = %.2f".format(meanNN))
println("  最小 (min)  = %.2f".format(minNN))
println("  最大 (max)  = %.2f".format(maxNN))

最近傍距離 [km]:
  平均 (mean) = 15.25
  最小 (min)  = 0.17
  最大 (max)  = 665.86


In [24]:
// --- 距離ビンの設計 ---
// rMin: 平均最近傍距離の 1/2（クラスタリングスケールの下限を確実に捉える）
// rMax: 大陸スケール上限 [km]
// nBins: Δln(r) ≈ 0.15 を目標に逆算
val rMin  = meanNN / 2.0
val rMax  = 2000.0
val dLogR = 0.15  // 対数ビン幅の目標値
val nBins = kotlin.math.ceil(kotlin.math.ln(rMax / rMin) / dLogR).toInt()

val bins = DoubleArray(nBins + 1) { i ->
    rMin * (rMax / rMin).pow(i.toDouble() / nBins)
}
val rCenters = DoubleArray(nBins) { i -> kotlin.math.sqrt(bins[i] * bins[i + 1]) }

println("rMin  = %.2f km  (= meanNN / 2)".format(rMin))
println("rMax  = %.1f km".format(rMax))
println("nBins = %d  (Δln r = %.3f)".format(nBins, kotlin.math.ln(rMax / rMin) / nBins))
println("ビン端点: ${bins.take(4).map { "%.1f".format(it) }} ... ${"%.1f".format(bins.last())} km")

rMin  = 7.63 km  (= meanNN / 2)
rMax  = 2000.0 km
nBins = 38  (Δln r = 0.147)
ビン端点: [7.6, 8.8, 10.2, 11.8] ... 2000.0 km


In [25]:
// --- ランダムカタログ生成（オーストラリア・バウンディングボックス） ---
// 注: 大陸マスクなしの矩形一様サンプリング。海洋点を含むが
//     Landy-Szalay推定量はこの境界効果を RR で相殺する。
val rng = Random(42)
val latMin = -44.0; val latMax = -10.0
val lonMin = 113.0; val lonMax = 154.0
val nR = nD * 10   // N_R >> N_D で RR のショットノイズを抑制

val randomPoints = List(nR) {
    Point(
        lat = latMin + rng.nextDouble() * (latMax - latMin),
        lon = lonMin + rng.nextDouble() * (lonMax - lonMin)
    )
}
println("N_R = $nR")

N_R = 6850


In [26]:
// --- ペアカウント関数 ---
// points2 == null のとき自己ペア（i < j のみ）をカウント
fun pairCounts(
    points1: List<Point>,
    points2: List<Point>?,
    bins: DoubleArray
): LongArray {
    val counts = LongArray(bins.size - 1)
    val pts2 = points2 ?: points1
    val selfPairs = points2 == null

    for (i in points1.indices) {
        val jStart = if (selfPairs) i + 1 else 0
        for (j in jStart until pts2.size) {
            val d = haversine(points1[i], pts2[j])
            // 二分探索でビンを特定
            var lo = 0; var hi = bins.size - 1
            while (lo < hi - 1) {
                val mid = (lo + hi) / 2
                if (bins[mid] <= d) lo = mid else hi = mid
            }
            if (d >= bins[lo] && d < bins[hi]) counts[lo]++
        }
    }
    return counts
}

// --- DD, DR, RR の計算（※ RR は N_R^2/2 ≈ 2.3×10^7 ペアで数分かかる） ---
println("DD を計算中...")
val nDD = pairCounts(dataPoints, null, bins)

println("DR を計算中...")
val nDR = pairCounts(dataPoints, randomPoints, bins)

println("RR を計算中...")
val nRR = pairCounts(randomPoints, null, bins)

println("完了")

DD を計算中...
DR を計算中...
RR を計算中...
完了


In [27]:
// --- Landy-Szalay 推定量 ---
val normDD = nD.toLong() * (nD - 1) / 2
val normDR = nD.toLong() * nR
val normRR = nR.toLong() * (nR - 1) / 2

data class XiBin(val rCenter: Double, val xi: Double, val xiErr: Double, val nRR: Long)

val xiResult: List<XiBin> = (0 until nBins).map { i ->
    val dd = nDD[i].toDouble() / normDD
    val dr = nDR[i].toDouble() / normDR
    val rr = nRR[i].toDouble() / normRR

    val xi = if (rr > 0.0) (dd - 2.0 * dr + rr) / rr else Double.NaN
    // Poisson誤差の下限: σ_ξ ≈ (1 + ξ) / sqrt(n_RR)
    val xiErr = if (nRR[i] > 0) (1.0 + xi) / sqrt(nRR[i].toDouble()) else Double.NaN

    XiBin(rCenters[i], xi, xiErr, nRR[i])
}

xiResult.forEach { b ->
    println("r = %7.1f km  ξ = %+.4f ± %.4f  (n_RR = %d)".format(b.rCenter, b.xi, b.xiErr, b.nRR))
}

r =     8.2 km  ξ = +519.0469 ± 54.5157  (n_RR = 91)
r =     9.5 km  ξ = +553.9043 ± 51.7451  (n_RR = 115)
r =    11.0 km  ξ = +400.6769 ± 29.2953  (n_RR = 188)
r =    12.7 km  ξ = +377.3139 ± 24.4711  (n_RR = 239)
r =    14.8 km  ξ = +388.7789 ± 22.9282  (n_RR = 289)
r =    17.1 km  ξ = +299.9277 ± 14.2975  (n_RR = 443)
r =    19.8 km  ξ = +258.2750 ± 10.6114  (n_RR = 597)
r =    22.9 km  ξ = +219.4793 ± 7.7903  (n_RR = 801)
r =    26.5 km  ξ = +192.8156 ± 6.1107  (n_RR = 1006)
r =    30.7 km  ξ = +148.3290 ± 4.0597  (n_RR = 1353)
r =    35.5 km  ξ = +114.6171 ± 2.7829  (n_RR = 1726)
r =    41.1 km  ξ = +82.6249 ± 1.7081  (n_RR = 2397)
r =    47.6 km  ξ = +64.8839 ± 1.1670  (n_RR = 3187)
r =    55.2 km  ξ = +46.4374 ± 0.7242  (n_RR = 4291)
r =    63.9 km  ξ = +31.8095 ± 0.4270  (n_RR = 5904)
r =    73.9 km  ξ = +21.2763 ± 0.2531  (n_RR = 7748)
r =    85.6 km  ξ = +14.7309 ± 0.1554  (n_RR = 10243)
r =    99.1 km  ξ = +9.9995 ± 0.0941  (n_RR = 13666)
r =   114.8 km  ξ = +9.7050 ± 0.0793

In [28]:
// --- 可視化 ---
// エラーバー: geomLineRange（縦線）+ geomRibbon（帯）の二重表示
// Poisson 誤差 σ_ξ ≈ (1+ξ) / √n_RR を ±1σ で描画
val plotData = mapOf(
    "r"      to xiResult.map { it.rCenter },
    "xi"     to xiResult.map { it.xi },
    "xiLow"  to xiResult.map { it.xi - it.xiErr },
    "xiHigh" to xiResult.map { it.xi + it.xiErr }
)

letsPlot(plotData) +
    geomRibbon(alpha = 0.20, fill = "#4682B4") {
        x = "r"; ymin = "xiLow"; ymax = "xiHigh"
    } +
    geomLineRange(color = "#4682B4", size = 0.6, alpha = 0.8) {
        x = "r"; ymin = "xiLow"; ymax = "xiHigh"
    } +
    geomLine(color = "#4682B4", size = 1.0) { x = "r"; y = "xi" } +
    geomPoint(color = "#4682B4", size = 2.0) { x = "r"; y = "xi" } +
    geomHLine(yintercept = 0.0, linetype = "dashed", color = "#808080") +
    scaleXLog10(name = "分離距離 r [km]") +
    scaleYContinuous(name = "ξ(r)") +
    ggtitle(
        "Coles店舗の2点相関関数（Landy-Szalay推定量）",
        "エラーバー: Poisson 誤差 ±1σ ≈ (1+ξ)/√n_RR"
    ) +
    theme(plotBackground = elementRect(fill = "#ffffff"))

<path d="M0.0 30.03700439399313 L0.0 30.03700439399313 L14.305842150250356 14.590909090909065 L28.611684300500713 99.15940507047418 L42.91752645075107 112.72832440034387 L57.223368601001425 107.95198107656421 L71.52921075125181 154.87840230639102 L85.83505290150214 176.7038604384051 L100.14089505175252 196.7376296622292 L114.44673720200285 210.3816810645531 L128.75257935225324 232.78422696872494 L143.05842150250362 249.62724488727568 L157.364263652754 265.5452630259515 L171.67010580300433 274.345982420587 L185.97594795325472 283.43905652607623 L200.28179010350505 290.6237768572982 L214.58763225375543 295.7780305343129 L228.89347440400576 298.97591414893856 L243.1993165542561 301.2830464708266 L257.5051587045064 301.4319681869373 L271.8110008547568 302.9490054022828 L286.1168430050072 303.66250324459384 L300.42268515525757 304.80115806346225 L314.72852730550784 305.20466214276394 L329.0343694557583 304.6744466556769 L343.3402116060086 305.6253127728673 L357.64605375625894 305.8285166779695 L371.9518959065094 305.7400083986213 L386.2577380567597 305.6487584302335 L400.56358020701003 305.9144695106442 L414.8694223572605 305.7724288165079 L429.1752645075108 303.80832707149057 L443.481106657761 305.20982877756705 L457.7869488080116 306.1006536381397 L472.0927909582619 306.2430667563415 L486.39863310851223 306.09346846268426 L500.70447525876256 305.84987005176066 L515.0103174090129 306.26767663208864 L529.3161595592633 306.40880673332856 L529.3161595592633 306.4090909090909 L515.0103174090129 306.26818652898726 L500.70447525876256 305.8510868688652 L486.39863310851223 306.0943922372495 L472.0927909582619 306.243808708849 L457.7869488080116 306.10180082165346 L443.481106657761 305.2133351491344 L429.1752645075108 303.8162367117407 L414.8694223572605 305.7751528731327 L400.56358020701003 305.9170555406174 L386.2577380567597 305.6528261603295 L371.9518959065094 305.7442361082769 L357.64605375625894 305.83288041682533 L343.3402116060086 305.6316033184835 L329.0343694557583 304.6885227169148 L314.72852730550784 305.21644910153316 L300.42268515525757 304.81857606735457 L286.1168430050072 303.6950996563531 L271.8110008547568 302.99574773951565 L257.5051587045064 301.50830192378584 L243.1993165542561 301.3736359027514 L228.89347440400576 299.125559515171 L214.58763225375543 296.02168379440536 L200.28179010350505 291.03487932234884 L185.97594795325472 284.1362690602645 L171.67010580300433 275.46958237482204 L157.364263652754 267.18973024823265 L143.05842150250362 252.30656865998392 L128.75257935225324 236.69280237132114 L114.44673720200285 216.26487378948218 L100.14089505175252 204.23786525343996 L85.83505290150214 186.9202350894082 L71.52921075125181 168.6436381039771 L57.223368601001425 130.02657973894895 L42.91752645075107 136.28840509431217 L28.611684300500713 127.36409591556108 L14.305842150250356 64.40962453435864 L0.0 82.52321313775809 Z" fill="rgb(70,130,180)" stroke-width="1.0" fill-opacity="0.25098039215686274">
 
 
 
 <path d="M0.0 30.03700439399313 L0.0 30.03700439399313 L14.305842150250356 14.590909090909065 L28.611684300500713 99.15940507047418 L42.91752645075107 112.72832440034387 L57.223368601001425 107.95198107656421 L71.52921075125181 154.87840230639102 L85.83505290150214 176.7038604384051 L100.14089505175252 196.7376296622292 L114.44673720200285 210.3816810645531 L128.75257935225324 232.78422696872494 L143.05842150250362 249.62724488727568 L157.364263652754 265.5452630259515 L171.67010580300433 274.345982420587 L185.97594795325472 283.43905652607623 L200.28179010350505 290.6237768572982 L214.58763225375543 295.7780305343129 L228.89347440400576 298.97591414893856 L243.1993165542561 301.2830464708266 L257.5051587045064 301.4319681869373 L271.8110008547568 302.9490054022828 L286.1168430050072 303.66250324459384 L300.42268515525757 304.80115806346225 L314.72852730550784 305.20466214276394 L329.0343694557583 304.6744466556769 L343.3402116060086 305.6253127728673 L357.64605375625894 305.8285166779695 L371.9518959065094 305.74000

## 大陸マスク付きランダムカタログによる比較検証

現在のランダムカタログはバウンディングボックス内の一様サンプリング（海洋点 ~45% を含む）。
この幾何学的バイアスが特に中〜大スケール（$r > 300$ km）の $\xi(r)$ に系統誤差を与えている可能性がある。

とくに $r \approx 725$ km のバンプが「Sydney–Melbourne / Melbourne–Adelaide の実シグナル」か
「矩形カタログの幾何アーティファクト」かを判定するため、
オーストラリア本土ポリゴンでマスクした random catalog と比較する。

### テスト手順
1. オーストラリア大陸の簡略化ポリゴン（~37頂点）を定義し、Ray-casting 法で陸上判定
2. マスク済みランダムカタログ（$N_R^{\rm masked} = 10\,N_D$）を生成
3. $DD$ は共通、$DR^{\rm masked}$ と $RR^{\rm masked}$ のみ再計算
4. $\hat{\xi}^{\rm masked}(r)$ を元の $\hat{\xi}(r)$ と重ねてプロット

In [29]:
// --- オーストラリア大陸マスクポリゴンを CSV から読み込む ---
// australia_mask_polygon.csv: lat, lon の 37 頂点（時計回り）
val polyDf = DataFrame.readCSV("./output/australia_mask_polygon.csv")
val australiaPolygon: List<Pair<Double, Double>> = polyDf.rows().map { row ->
    Pair(row["lat"] as Double, row["lon"] as Double)
}
println("ポリゴン頂点数: ${australiaPolygon.size}")

// --- Ray-casting 法によるポリゴン内判定 ---
fun isInsidePolygon(lat: Double, lon: Double, poly: List<Pair<Double, Double>>): Boolean {
    var inside = false
    var j = poly.size - 1
    for (i in poly.indices) {
        val (latI, lonI) = poly[i]
        val (latJ, lonJ) = poly[j]
        if ((latI > lat) != (latJ > lat)) {
            val lonCross = (lonJ - lonI) * (lat - latI) / (latJ - latI) + lonI
            if (lon < lonCross) inside = !inside
        }
        j = i
    }
    return inside
}

fun isTasmania(lat: Double, lon: Double): Boolean =
    lat in -44.0..-40.0 && lon in 144.0..148.5

fun isOnContinent(lat: Double, lon: Double): Boolean =
    isInsidePolygon(lat, lon, australiaPolygon) || isTasmania(lat, lon)

// 検証: データ点の陸上判定率
val dataOnLand = dataPoints.count { isOnContinent(it.lat, it.lon) }
println("データ点 $nD 個中 $dataOnLand 個が陸上判定")
println("陸上判定率: ${"%.1f".format(dataOnLand.toDouble() / nD * 100)} %")

データ点 685 個中 650 個が陸上判定
陸上判定率: 94.9 %


In [30]:
// --- マスク済みランダムカタログの生成 (rejection sampling) ---
val rngMasked = Random(42)
val randomMasked: List<Point> = generateSequence {
    Point(
        lat = latMin + rngMasked.nextDouble() * (latMax - latMin),
        lon = lonMin + rngMasked.nextDouble() * (lonMax - lonMin)
    )
}.filter { isOnContinent(it.lat, it.lon) }
 .take(nR)
 .toList()

// f_land を推定（acceptance rate）
val rngEst = Random(99)
val trials = 100_000
val accepted = (0 until trials).count {
    isOnContinent(
        latMin + rngEst.nextDouble() * (latMax - latMin),
        lonMin + rngEst.nextDouble() * (lonMax - lonMin)
    )
}
val fLandEst = accepted.toDouble() / trials
println("マスク済みカタログ: N_R = ${randomMasked.size}")
println("f_land 推定値 = ${"%.3f".format(fLandEst)}  （海洋点の割合 ≈ ${"%.1f".format((1 - fLandEst) * 100)} %）")

マスク済みカタログ: N_R = 6850
f_land 推定値 = 0.513  （海洋点の割合 ≈ 48.7 %）


### マスク検証プロット

ランダムサンプル 3000 点を陸海に色分けし、ポリゴン境界と Coles 店舗を重ねて描画する。
- **緑**: 陸上（採用） — マスク済みカタログに使われる点
- **青**: 海洋（除外） — 矩形カタログにのみ含まれる点
- **赤**: Coles 店舗
- **黒線**: マスクポリゴン境界

店舗が黒線の内側に収まっていれば、ポリゴンの精度が確認できる。

In [ ]:
import org.jetbrains.letsPlot.coord.*

// ランダムサンプルを生成して陸海判定
val rngViz = Random(777)
val vizN = 3000
val vizLats = List(vizN) { latMin + rngViz.nextDouble() * (latMax - latMin) }
val vizLons = List(vizN) { lonMin + rngViz.nextDouble() * (lonMax - lonMin) }
val vizKind = (vizLats zip vizLons).map { (la, lo) ->
    if (isOnContinent(la, lo)) "陸上（採用）" else "海洋（除外）"
}
val maskVizData = mapOf("lat" to vizLats, "lon" to vizLons, "kind" to vizKind)

// ポリゴン境界
val polyData = mapOf(
    "lat" to australiaPolygon.map { it.first },
    "lon" to australiaPolygon.map { it.second }
)

// Coles 店舗
val storeData = mapOf(
    "lat" to dataPoints.map { it.lat },
    "lon" to dataPoints.map { it.lon }
)

letsPlot() +
    geomPoint(data = maskVizData, alpha = 0.35, size = 0.8) {
        x = "lon"; y = "lat"; color = "kind"
    } +
    geomPath(data = polyData, color = "#111111", size = 1.0) {
        x = "lon"; y = "lat"
    } +
    geomPoint(data = storeData, color = "#CC2222", size = 1.5, alpha = 0.8) {
        x = "lon"; y = "lat"
    } +
    scaleColorManual(
        values = mapOf("陸上（採用）" to "#2CA02C", "海洋（除外）" to "#4682B4")
    ) +
    coordMap(xlim = Pair(110.0, 155.0), ylim = Pair(-45.0, -10.0)) +
    ggtitle(
        "大陸マスク検証：ランダム点の陸海分類 vs Coles 店舗分布",
        "赤=Coles店舗、黒線=マスクポリゴン境界。店舗が黒線内に収まっていれば妥当"
    ) +
    theme(plotBackground = elementRect(fill = "#ffffff"))

In [31]:
// --- マスク済みカタログで DR・RR を再計算 ---
// DD は変わらないため nDD を再利用
val nRMasked = randomMasked.size

println("DR_masked を計算中...")
val nDR_masked = pairCounts(dataPoints, randomMasked, bins)

println("RR_masked を計算中...")
val nRR_masked = pairCounts(randomMasked, null, bins)

println("完了")

// --- マスク済み Landy-Szalay 推定量 ---
val normDR_masked = nD.toLong() * nRMasked
val normRR_masked = nRMasked.toLong() * (nRMasked - 1) / 2

val xiMasked: List<XiBin> = (0 until nBins).map { i ->
    val dd = nDD[i].toDouble() / normDD
    val dr = nDR_masked[i].toDouble() / normDR_masked
    val rr = nRR_masked[i].toDouble() / normRR_masked
    val xi = if (rr > 0.0) (dd - 2.0 * dr + rr) / rr else Double.NaN
    val xiErr = if (nRR_masked[i] > 0) (1.0 + xi) / sqrt(nRR_masked[i].toDouble()) else Double.NaN
    XiBin(rCenters[i], xi, xiErr, nRR_masked[i])
}

println("\n--- マスクあり vs マスクなし ---")
println("%-10s  %-12s  %-12s  %-10s".format("r [km]", "ξ (矩形)", "ξ (マスク)", "Δξ"))
xiResult.zip(xiMasked).forEach { (orig, masked) ->
    val delta = masked.xi - orig.xi
    println("%-10.1f  %-12.4f  %-12.4f  %-10.4f".format(
        orig.rCenter, orig.xi, masked.xi, delta))
}

DR_masked を計算中...
RR_masked を計算中...
完了

--- マスクあり vs マスクなし ---
r [km]      ξ (矩形)        ξ (マスク)       Δξ        
8.2         519.0469      246.3524      -272.6945 
9.5         553.9043      278.0353      -275.8690 
11.0        400.6769      224.7106      -175.9664 
12.7        377.3139      194.0434      -183.2705 
14.8        388.7789      188.4886      -200.2904 
17.1        299.9277      160.6057      -139.3220 
19.8        258.2750      148.8597      -109.4153 
22.9        219.4793      119.5387      -99.9406  
26.5        192.8156      101.1401      -91.6754  
30.7        148.3290      79.6359       -68.6931  
35.5        114.6171      56.1174       -58.4998  
41.1        82.6249       43.4912       -39.1337  
47.6        64.8839       33.8045       -31.0794  
55.2        46.4374       24.3387       -22.0987  
63.9        31.8095       17.0652       -14.7443  
73.9        21.2763       11.3014       -9.9750   
85.6        14.7309       7.8798        -6.8511   
99.1        9.9995 

In [32]:
// --- 比較プロット: 矩形カタログ vs 大陸マスク済みカタログ ---
// エラーバー: geomLineRange（縦線）+ geomRibbon（帯）
val compData = mapOf(
    "r"       to (xiResult.map { it.rCenter } + xiMasked.map { it.rCenter }),
    "xi"      to (xiResult.map { it.xi }      + xiMasked.map { it.xi }),
    "xiLow"   to (xiResult.map { it.xi - it.xiErr } + xiMasked.map { it.xi - it.xiErr }),
    "xiHigh"  to (xiResult.map { it.xi + it.xiErr } + xiMasked.map { it.xi + it.xiErr }),
    "catalog" to (List(nBins) { "矩形（海洋含む）" } + List(nBins) { "大陸マスク済み" })
)

letsPlot(compData) +
    geomRibbon(alpha = 0.12) {
        x = "r"; ymin = "xiLow"; ymax = "xiHigh"; fill = "catalog"
    } +
    geomLineRange(size = 0.5, alpha = 0.7) {
        x = "r"; ymin = "xiLow"; ymax = "xiHigh"; color = "catalog"
    } +
    geomLine(size = 1.2) { x = "r"; y = "xi"; color = "catalog" } +
    geomPoint(size = 2.0) { x = "r"; y = "xi"; color = "catalog" } +
    geomHLine(yintercept = 0.0, linetype = "dashed", color = "#808080") +
    geomVLine(xintercept = 725.0, linetype = "dotted", color = "#CC4444") +
    scaleXLog10(name = "分離距離 r [km]") +
    scaleYContinuous(name = "ξ(r)") +
    scaleColorManual(values = mapOf("矩形（海洋含む）" to "#4682B4", "大陸マスク済み" to "#E87040")) +
    scaleFillManual(values = mapOf("矩形（海洋含む）" to "#4682B4", "大陸マスク済み" to "#E87040")) +
    ggtitle(
        "ξ(r) 比較：ランダムカタログの幾何バイアス検証",
        "エラーバー: Poisson ±1σ。赤点線 = r=725 km（Melb–Adel）"
    ) +
    theme(plotBackground = elementRect(fill = "#ffffff"))

<path d="M0.0 28.25908429765201 L0.0 28.25908429765201 L10.89956039093164 13.727272727272748 L21.79912078186328 93.28999872703 L32.69868117279492 106.0557479371862 L43.59824156372656 101.56212545107178 L54.49780195465823 145.7108826355448 L65.39736234558984 166.24445042651286 L76.29692273652151 185.09238278045518 L87.19648312745312 197.92881685292596 L98.09604351838479 219.00531349419657 L108.99560390931643 234.85136117590616 L119.89516430024807 249.8271615724588 L130.7947246911797 258.1069496624034 L141.69428508211135 266.6617874323141 L152.593845473043 273.4212305142394 L163.49340586397463 278.2703941216501 L174.39296625490627 281.27898892829126 L185.2925266458379 283.4495577038778 L196.19208703676955 283.58966446062084 L207.0916474277012 285.01690536041326 L217.99120781863283 285.6881699096011 L228.89076820956453 286.7594255859917 L239.7903286004961 287.1390454920738 L250.6898889914278 286.64021446721875 L261.58944938235936 287.5347970120325 L272.48900977329106 287.72597289184995 L283.38857016422276 287.6427035857457 L294.28813055515434 287.5568548763862 L305.1876909460859 287.80683801302524 L316.08725133701773 287.6732049774902 L326.9868117279492 285.82536214707943 L337.8863721188808 287.1439063039449 L348.7859325098126 287.9820015482146 L359.6854929007442 288.115984963327 L370.58505329167576 287.97524167694814 L381.48461368260746 287.74606231886924 L392.38417407353904 288.1391381337633 L403.28373446447074 288.27191447612586 L403.28373446447074 288.2721818309848 L392.38417407353904 288.13961784889557 L381.48461368260746 287.7472071102571 L370.58505329167576 287.9761107715414 L359.6854929007442 288.1166829982556 L348.7859325098126 287.98308082775617 L337.8863721188808 287.1472051267162 L326.9868117279492 285.8328036006928 L316.08725133701773 287.67576779202693 L305.1876909460859 287.8092709709702 L294.28813055515434 287.56068182980806 L283.38857016422276 287.64668104923305 L272.48900977329106 287.73007833271066 L261.58944938235936 287.54071520829996 L250.6898889914278 286.65345733964375 L239.7903286004961 287.1501347583551 L228.89076820956453 286.77581258513476 L217.99120781863283 285.7188368778064 L207.0916474277012 285.06088092988233 L196.19208703676955 283.66147985853075 L185.2925266458379 283.53478497518284 L174.39296625490627 281.4197765009302 L163.49340586397463 278.4996250822713 L152.593845473043 273.807999043179 L141.69428508211135 267.3177306317134 L130.7947246911797 259.1640416002066 L119.89516430024807 251.37428965492524 L108.99560390931643 237.37209068307547 L98.09604351838479 222.68253289137368 L87.19648312745312 203.4637722208257 L76.29692273652151 192.1486651745671 L65.39736234558984 175.8560984480768 L54.49780195465823 158.66132796863943 L43.59824156372656 122.33008667098417 L32.69868117279492 128.22126268441926 L21.79912078186328 119.8252003245345 L10.89956039093164 60.59712373473104 L0.0 77.63853460802878 Z" fill="rgb(70,130,180)" stroke-width="1.0" fill-opacity="0.14901960784313725">
 
 
 
 <path d="M0.0 168.3441437030922 L0.0 168.3441437030922 L10.89956039093164 153.7317339624912 L21.79912078186328 180.65796099403775 L32.69868117279492 196.03499721249733 L43.59824156372656 199.13481269393725 L54.49780195465823 212.7341943738424 L65.39736234558984 218.4898299979614 L76.29692273652151 232.4555398090328 L87.19648312745312 241.15522560716545 L98.09604351838479 251.22333025670292 L108.99560390931643 262.1674771571188 L119.89516430024807 268.02321393237577 L130.7947246911797 272.5078732794718 L141.69428508211135 276.87021261105826 L152.593845473043 280.2133575584467 L163.49340586397463 282.8559018358147 L174.39296625490627 284.4230425241057 L185.2925266458379 285.5435161885993 L196.19208703676955 285.58610057461976 L207.0916474277012 286.32714064014004 L217.99120781863283 286.68730906537263 L228.89076820956453 287.2195340049943 L239.7903286004961 287.44820574398284 L250.6898889914278 287.16251463723194 L261.58944938235936 287.70529093839434 L272.48900977329106 287.8167225917194 L283.38857016422276 287.75421160